In [ ]:
import pandas as pd
from typing import List, Optional
from sqlalchemy import String, Integer, Date, ForeignKey, PrimaryKeyConstraint
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship
from sqlalchemy.orm import Session
from typing import Optional
from datetime import date



usuario = "thebridge"
password = "7UBu7Wvi9QX7onkaziGFLPvpuptrzLxe"
host = "dpg-d7agm5idbo4c73a7b800-a.oregon-postgres.render.com" # o la IP del servidor
puerto = "5432"
nombre_db = "thebridge_ivka"

engine = create_engine(f'postgresql://{usuario}:{password}@{host}/{nombre_db}')
# Prueba de conexión
try:
    with engine.connect() as connection:
        print("¡Conexión exitosa a PostgreSQL!")
except Exception as e:
    print(f"Error al conectar: {e}")

¡Conexión exitosa a PostgreSQL!


In [76]:
inspector = inspect(engine)
nombres_tablas = inspector.get_table_names()

for tabla in nombres_tablas:
    print(f"{tabla}")

campus
promocion
vertical
alumnos
profesores
proyectos
notas


In [80]:
class Base(DeclarativeBase):
    pass
class Campus(Base):
    __tablename__ = "campus"

    id_campus: Mapped[int] = mapped_column(Integer, primary_key=True)
    nombre_campus: Mapped[str] = mapped_column(String(25), nullable=False)

    # Relaciones
    promociones: Mapped[List["Promocion"]] = relationship(back_populates="campus")


class Vertical(Base):
    __tablename__ = "vertical"

    id_vertical: Mapped[int] = mapped_column(Integer, primary_key=True)
    nombre_vertical: Mapped[str] = mapped_column(String(25), nullable=False)

    # Relaciones
    promociones: Mapped[List["Promocion"]] = relationship(back_populates="vertical")
    proyectos: Mapped[List["Proyecto"]] = relationship(back_populates="vertical")


class Proyecto(Base):
    __tablename__ = "proyectos"

    id_proyecto: Mapped[int] = mapped_column(Integer, primary_key=True)
    nombre_proyecto: Mapped[str] = mapped_column(String(100), nullable=False)
    id_vertical: Mapped[Optional[int]] = mapped_column(ForeignKey("vertical.id_vertical"))

    # Relaciones
    vertical: Mapped["Vertical"] = relationship(back_populates="proyectos")
    notas: Mapped[List["Nota"]] = relationship(back_populates="proyecto")


class Promocion(Base):
    __tablename__ = "promocion"

    id_promocion: Mapped[int] = mapped_column(Integer, primary_key=True)
    nombre_mes: Mapped[str] = mapped_column(String(25), nullable=False)
    fecha_comienzo: Mapped[Optional[date]] = mapped_column(Date)
    id_campus: Mapped[Optional[int]] = mapped_column(ForeignKey("campus.id_campus"))
    id_vertical: Mapped[Optional[int]] = mapped_column(ForeignKey("vertical.id_vertical"))

    # Relaciones
    campus: Mapped["Campus"] = relationship(back_populates="promociones")
    vertical: Mapped["Vertical"] = relationship(back_populates="promociones")
    alumnos: Mapped[List["Alumno"]] = relationship(back_populates="promocion")
    profesores: Mapped[List["Profesor"]] = relationship(back_populates="promocion")


class Alumno(Base):
    __tablename__ = "alumnos"

    id_alumno: Mapped[int] = mapped_column(Integer, primary_key=True)
    nombre: Mapped[str] = mapped_column(String(150), nullable=False)
    email: Mapped[str] = mapped_column(String(150), nullable=False)
    id_promocion: Mapped[Optional[int]] = mapped_column(ForeignKey("promocion.id_promocion"))

    # Relaciones
    promocion: Mapped["Promocion"] = relationship(back_populates="alumnos")
    notas: Mapped[List["Nota"]] = relationship(back_populates="alumno")


class Profesor(Base):
    __tablename__ = "profesores"

    id_profesor: Mapped[int] = mapped_column(Integer, primary_key=True)
    nombre_profesor: Mapped[str] = mapped_column(String(50), nullable=False)
    rol: Mapped[str] = mapped_column(String(50), nullable=False)
    modalidad: Mapped[str] = mapped_column(String(50), nullable=False)
    id_promocion: Mapped[Optional[int]] = mapped_column(ForeignKey("promocion.id_promocion"))

    # Relaciones
    promocion: Mapped["Promocion"] = relationship(back_populates="profesores")


class Nota(Base):
    __tablename__ = "notas"

    id_alumno: Mapped[int] = mapped_column(ForeignKey("alumnos.id_alumno"), primary_key=True)
    id_proyecto: Mapped[int] = mapped_column(ForeignKey("proyectos.id_proyecto"), primary_key=True)
    nota: Mapped[str] = mapped_column(String(50), nullable=False)

    # Relaciones
    alumno: Mapped["Alumno"] = relationship(back_populates="notas")
    proyecto: Mapped["Proyecto"] = relationship(back_populates="notas")

In [ ]:

nombres_campus = ["Madrid", "Valencia", "Sevilla", "Barcelona"]

# 1. Convertimos la lista de strings en una lista de objetos Campus
objetos_campus = [Campus(nombre_campus=nombre) for nombre in nombres_campus]

# 2. Abrimos la sesión y añadimos todo de golpe
with Session(engine) as session:
    session.add_all(objetos_campus)
    session.commit()
    print(f"Se han añadido {len(objetos_campus)} registros.")

Se han añadido 4 registros.


TypeError: sessionmaker.__call__() takes 1 positional argument but 2 were given